In [10]:
import urllib.parse
from sqlalchemy import create_engine
import pandas as pd
import os
from google.colab import drive

# --- CONFIGURACIÓN DEL POOLER ---
USER = "postgres.gsptclcqrrnpomutfjpz" # <--- Usuario especial del pooler
PASSWORD = urllib.parse.quote_plus("5u&9VA*#UtA6FYz")
HOST = "aws-1-us-east-1.pooler.supabase.com" # <--- Host del pooler
PORT = "6543"
DBNAME = "postgres"

# Construcción de la URL
DB_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

try:
    engine = create_engine(DB_URL, pool_pre_ping=True)
    with engine.connect() as conn:
        print("✅ ¡CONEXIÓN EXITOSA! El túnel IPv4 del Pooler está funcionando.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ ¡CONEXIÓN EXITOSA! El túnel IPv4 del Pooler está funcionando.


In [16]:
from sqlalchemy import text

# 1. Montar Drive (si no está montado)
drive.mount('/content/drive')

# 2. Ruta de tus archivos
PATH_DRIVE = '/content/drive/MyDrive/SEMESTRE 4/Proyecto productivo IIB/IDL01/Bronce_Data raw /'

def ejecutar_pipeline_bronce():
    try:
        # Usamos text() también aquí para la consulta inicial
        query_config = text("SELECT nombre_tabla FROM meta.pipeline_config WHERE estado = 'ACTIVE'")
        tablas_activas = pd.read_sql(query_config, engine)['nombre_tabla'].tolist()
        print(f"📋 Tablas encontradas en metadatos: {len(tablas_activas)}")
    except Exception as e:
        print(f"❌ Error al leer metadatos: {e}")
        return

    for tabla in tablas_activas:
        # Usamos la ruta con el espacio al final que descubrimos antes
        archivo_path = os.path.join(PATH_DRIVE, f"{tabla}.csv")

        if os.path.exists(archivo_path):
            print(f"⌛ Subiendo {tabla}...")
            df = pd.read_csv(archivo_path)
            df.columns = [c.lower().replace(' ', '_').replace('.', '_') for c in df.columns]
            df['etl_fecha_carga'] = pd.Timestamp.now()

            # Cargar datos
            df.to_sql(tabla, engine, schema='bronce', if_exists='replace', index=False)

            # --- CAMBIO AQUÍ: Usar text() y commit() ---
            with engine.connect() as conn:
                # Envolvemos el UPDATE en text()
                statement = text(f"UPDATE meta.pipeline_config SET ultima_carga = NOW() WHERE nombre_tabla = '{tabla}'")
                conn.execute(statement)
                conn.commit() # Importante para que el Pooler guarde los cambios

            print(f"   ✅ {tabla} lista en Bronce.")
        else:
            print(f"   ⚠️ No se encontró el archivo: {archivo_path}")

# Ejecutar
ejecutar_pipeline_bronce()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📋 Tablas encontradas en metadatos: 9
⌛ Subiendo ads_campanas_maestro...
   ✅ ads_campanas_maestro lista en Bronce.
⌛ Subiendo ads_insights_diario...
   ✅ ads_insights_diario lista en Bronce.
⌛ Subiendo clima_diario_log...
   ✅ clima_diario_log lista en Bronce.
⌛ Subiendo sap_canales_maestro...
   ✅ sap_canales_maestro lista en Bronce.
⌛ Subiendo sap_clientes_maestro...
   ✅ sap_clientes_maestro lista en Bronce.
⌛ Subiendo sap_inventario_diario...
   ✅ sap_inventario_diario lista en Bronce.
⌛ Subiendo sap_productos_maestro...
   ✅ sap_productos_maestro lista en Bronce.
⌛ Subiendo sap_ventas_cabecera...
   ✅ sap_ventas_cabecera lista en Bronce.
⌛ Subiendo sap_ventas_detalle...
   ✅ sap_ventas_detalle lista en Bronce.
